<a href="https://colab.research.google.com/github/epkalibbala/quant-finance-research-lab/blob/main/pd_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Import Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

import matplotlib.pyplot as plt

In [2]:
# Create Simulated Credit Dataset
np.random.seed(42)

n = 2000

data = pd.DataFrame({

    "income": np.random.normal(50000,15000,n),
    "loan_amount": np.random.normal(15000,5000,n),
    "interest_rate": np.random.uniform(0.05,0.25,n),
    "dti": np.random.uniform(0.1,0.7,n),
    "credit_score": np.random.normal(650,50,n)
})

# default probability rule
prob_default = (

    0.002*data["loan_amount"]
    -0.003*data["income"]
    +3*data["dti"]
    +0.04*data["interest_rate"]
    -0.002*data["credit_score"]
)

prob_default = 1/(1+np.exp(-prob_default))

data["default"] = np.random.binomial(1,prob_default)

In [3]:
# Prepare Data
X = data[[

    "income",
    "loan_amount",
    "interest_rate",
    "dti",
    "credit_score"
]]

y = data["default"]

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size=0.2,
    random_state=42
)

In [4]:
# Train Logistic Regression Model
model = LogisticRegression(max_iter=1000)

model.fit(X_train,y_train)

pd_predictions = model.predict_proba(X_test)[:,1]

In [5]:
# Evaluate Model
auc = roc_auc_score(y_test,pd_predictions)

print("ROC AUC:",auc)

ROC AUC: 1.0


In [6]:
# Confusion Matrix
pred_class = (pd_predictions > 0.5).astype(int)

print(confusion_matrix(y_test,pred_class))

[[398   0]
 [  0   2]]


In [7]:
# Classification Report
print(classification_report(y_test,pred_class))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       398
           1       1.00      1.00      1.00         2

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400



In [8]:
# Interpret Model Coefficients
coefficients = pd.DataFrame({

    "Feature":X.columns,
    "Coefficient":model.coef_[0]
})

print(coefficients)

         Feature   Coefficient
0         income -1.605556e-02
1    loan_amount  1.111680e-02
2  interest_rate -5.897086e-07
3            dti -1.182025e-05
4   credit_score -2.937926e-02


In [9]:
# Compute Expected Loss
LGD = 0.6

EAD = X_test["loan_amount"]

expected_loss = pd_predictions * LGD * EAD

portfolio_loss = expected_loss.sum()

print("Expected Portfolio Loss:",portfolio_loss)

Expected Portfolio Loss: 32371.76694288759
